Reto: "Predicción de Fallos en Equipos en Plantas Energéticas a 15 días vista"
Objetivo del Reto:
Evolucionar la solución para desarrollar un modelo analítico predictivo que permita anticipar fallos a 15 días vista en los equipos de la planta energética con una precisión mínima del 80% (accuracy ≥ 0.80) y por lo menos un 0.75 de f1.

Por un lado se quiere predecir las horas operativas que tendrá un equipo en los próximos 15 días, de cara a considerar su mantenimiento conforme a las horas recomendadas de revisión.
Por otro lado, se quiere evaluar si una máquina fallará en los próximos 15 días, con un 80% de fiabilidad.
Evalúa la robustés del modelo en el tiempo.

Para lograr esto, deberéis:

Mezclar y preprocesar los datos de los 3 datasets proporcionados, teniendo en cuenta la componente temporal.
Seleccionar variables relevantes y generar nuevas características si es necesario. Considera usar PCA para reducir dimensiones (conservando 95% de la varianza).
Construir y evaluar múltiples modelos predictivos, que inclyan forecasting y comparando su rendimiento.
Ajustar hiperparámetros y optimizar el modelo para alcanzar la precisión deseada.
Documentar los hallazgos y justificar la elección del mejor modelo.
Bonus 1: Encontrar patrones de fallos
Usa K-Means o DBSCAN para agrupar los equipos según sus condiciones operativas y comportamiento previo a los fallos.
Incluye las siguientes características para el clustering:
Promedio de temperatura y vibración por equipo.
Desviación estándar de temperatura y vibración.
Horas operativas acumuladas promedio.
Visualiza los clusters utilizando gráficos de dispersión o mapas de calor.
Interpreta los resultados: ¿Qué patrones emergen? ¿Hay grupos de equipos más propensos a fallar?

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate
from sklearn.feature_selection import SelectKBest, f_classif


In [19]:
#Carga de datos
caracteristicas_equipos = pd.read_csv('../caso_practico_4/data/Caracteristicas_Equipos.csv')
historicos_ordenes = pd.read_csv('../caso_practico_4/data/Historicos_Ordenes.csv')
registros_condiciones = pd.read_csv('../caso_practico_4/data/Registros_Condiciones.csv')
caracteristicas_equipos.info()
historicos_ordenes.info()
registros_condiciones.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 6 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   ID_Equipo                    500 non-null    int64 
 1   Tipo_Equipo                  500 non-null    object
 2   Fabricante                   500 non-null    object
 3   Modelo                       500 non-null    object
 4   Potencia_kW                  500 non-null    int64 
 5   Horas_Recomendadas_Revision  500 non-null    int64 
dtypes: int64(3), object(3)
memory usage: 23.6+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31866 entries, 0 to 31865
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   ID_Orden             31866 non-null  int64 
 1   ID_Equipo            31866 non-null  int64 
 2   Fecha                31866 non-null  object
 3   Tipo_Mantenimiento   31866 non-

In [20]:
#Pasamos a datetime las fechas

historicos_ordenes['Fecha'] = pd.to_datetime(historicos_ordenes['Fecha'])
registros_condiciones['Fecha'] = pd.to_datetime(registros_condiciones['Fecha'])
historicos_ordenes.info()
registros_condiciones.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31866 entries, 0 to 31865
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   ID_Orden             31866 non-null  int64         
 1   ID_Equipo            31866 non-null  int64         
 2   Fecha                31866 non-null  datetime64[ns]
 3   Tipo_Mantenimiento   31866 non-null  object        
 4   Costo_Mantenimiento  31866 non-null  int64         
 5   Duracion_Horas       31866 non-null  int64         
 6   Ubicacion            31866 non-null  object        
dtypes: datetime64[ns](1), int64(4), object(2)
memory usage: 1.7+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 730500 entries, 0 to 730499
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   ID_Registro       730500 non-null  int64         
 1   ID_Equipo         730500 non-n

In [21]:
#Creamos un solo df con informacion de los 3, ordenados por ID_Equipo y Fecha
def mezclar_datasets(caracteristicas_equipos, historicos_ordenes, registros_condiciones):
    merged_data = pd.merge(caracteristicas_equipos, historicos_ordenes, on='ID_Equipo', how='outer')
    merged_data = pd.merge(merged_data, registros_condiciones, on=['ID_Equipo', 'Fecha'], how='outer')
    merged_data = merged_data.sort_values(by=['ID_Equipo', 'Fecha'])
    merged_data = merged_data.fillna(method='ffill')
    return merged_data

In [22]:
merged_df = mezclar_datasets(caracteristicas_equipos, historicos_ordenes, registros_condiciones)
merged_df

/tmp/ipykernel_12645/781652571.py:6: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged_data = merged_data.fillna(method='ffill')


,ID_Equipo,Tipo_Equipo,Fabricante,Modelo,Potencia_kW,Horas_Recomendadas_Revision,ID_Orden,Fecha,Tipo_Mantenimiento,Costo_Mantenimiento,Duracion_Horas,Ubicacion,ID_Registro,Temperatura_C,Vibracion_mm_s,Horas_Operativas
0,1,NaN,NaN,NaN,NaN,NaN,NaN,2021-01-01,NaN,NaN,NaN,NaN,1,36.587104,4.430381,579
1,1,NaN,NaN,NaN,NaN,NaN,NaN,2021-01-02,NaN,NaN,NaN,NaN,740,24.495730,3.871645,592
2,1,NaN,NaN,NaN,NaN,NaN,NaN,2021-01-03,NaN,NaN,NaN,NaN,1101,54.798241,6.397340,611
3,1,NaN,NaN,NaN,NaN,NaN,NaN,2021-01-04,NaN,NaN,NaN,NaN,1762,82.709427,7.051379,622
4,1,NaN,NaN,NaN,NaN,NaN,NaN,2021-01-05,NaN,NaN,NaN,NaN,2344,75.120966,0.861798,624
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
730732,500,Motor,Fabricante_C,Modelo_8,299.0,710.0,31228.0,2024-12-27,Preventivo,853.0,1.0,Ubicacion_D,728405,47.044287,7.947597,1647
730733,500,Motor,Fabricante_C,Modelo_8,299.0,710.0,31228.0,2024-12-28,Preventivo,853.0,1.0,Ubicacion_D,728934,75.447939,3.135369,1659
730734,500,Motor,Fabricante_C,Modelo_8,299.0,710.0,31814.0,2024-12-29,Preventivo,491.0,1.0,Ubicacion_D,729051,62.975469,0.413056,1677
730735,500,Motor,Fabricante_C,Modelo_8,299.0,710.0,31814.0,2024-12-30,Preventivo,491.0,1.0,Ubicacion_D,729629,71.272542,0.545360,1683


Preprocesamos datos

In [23]:
#Dataset sin nulos
def preprocess_data (df):
    #Eliminamos nulos
    df=df.dropna()
    #Eliminamos duplicados
    df = df.drop_duplicates()
    #Añadimos columnas de día, mes, año
    df['Dia'] = df['Fecha'].dt.day
    df['Quincena'] = df['Fecha'].dt.day.apply(lambda x: (x - 1) // 15 + 1)
    df['Mes'] = df['Fecha'].dt.month
    df['Año'] = df['Fecha'].dt.year
    #Añadimos columna de Horas_Hasta_Revision
    df['Horas_Hasta_Revision']=df['Horas_Recomendadas_Revision']-df['Horas_Operativas']
    #Cambiamos las variables tipo object a numéricas ordinales para estudios posteriores 
    for column in df.select_dtypes(include=['object']).columns:
        df[column] = df[column].astype('category').cat.codes

    return df

In [24]:
info_equipos = preprocess_data(merged_df)
info_equipos

,ID_Equipo,Tipo_Equipo,Fabricante,Modelo,Potencia_kW,Horas_Recomendadas_Revision,ID_Orden,Fecha,Tipo_Mantenimiento,Costo_Mantenimiento,...,Ubicacion,ID_Registro,Temperatura_C,Vibracion_mm_s,Horas_Operativas,Dia,Quincena,Mes,Año,Horas_Hasta_Revision
29,1,0,0,8,432.0,657.0,391.0,2021-01-30,1,964.0,...,0,14588,84.711721,0.912324,895,30,2,1,2021,-238.0
30,1,0,0,8,432.0,657.0,391.0,2021-01-31,1,964.0,...,0,15270,87.295714,2.676511,901,31,3,1,2021,-244.0
31,1,0,0,8,432.0,657.0,391.0,2021-02-01,1,964.0,...,0,15514,86.196544,6.886018,918,1,1,2,2021,-261.0
32,1,0,0,8,432.0,657.0,391.0,2021-02-02,1,964.0,...,0,16369,57.088007,7.776112,938,2,1,2,2021,-281.0
33,1,0,0,8,432.0,657.0,391.0,2021-02-03,1,964.0,...,0,16759,94.758073,4.335325,962,3,1,2,2021,-305.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
730732,500,3,2,8,299.0,710.0,31228.0,2024-12-27,1,853.0,...,3,728405,47.044287,7.947597,1647,27,2,12,2024,-937.0
730733,500,3,2,8,299.0,710.0,31228.0,2024-12-28,1,853.0,...,3,728934,75.447939,3.135369,1659,28,2,12,2024,-949.0
730734,500,3,2,8,299.0,710.0,31814.0,2024-12-29,1,491.0,...,3,729051,62.975469,0.413056,1677,29,2,12,2024,-967.0
730735,500,3,2,8,299.0,710.0,31814.0,2024-12-30,1,491.0,...,3,729629,71.272542,0.545360,1683,30,2,12,2024,-973.0


In [34]:
info_equipos.info()

<class 'pandas.core.frame.DataFrame'>
Index: 730708 entries, 29 to 730736
Data columns (total 21 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   ID_Equipo                    730708 non-null  int64         
 1   Tipo_Equipo                  730708 non-null  int8          
 2   Fabricante                   730708 non-null  int8          
 3   Modelo                       730708 non-null  int8          
 4   Potencia_kW                  730708 non-null  float64       
 5   Horas_Recomendadas_Revision  730708 non-null  float64       
 6   ID_Orden                     730708 non-null  float64       
 7   Fecha                        730708 non-null  datetime64[ns]
 8   Tipo_Mantenimiento           730708 non-null  int8          
 9   Costo_Mantenimiento          730708 non-null  float64       
 10  Duracion_Horas               730708 non-null  float64       
 11  Ubicacion                    7

In [26]:
#Agregamos datos de forma quincenal
info_equipos_agg = info_equipos.groupby(['ID_Equipo','Tipo_Equipo','Fabricante','Modelo','Potencia_kW','Horas_Recomendadas_Revision','Tipo_Mantenimiento', 'Quincena']).agg({
 'Costo_Mantenimiento': 'sum', 'Temperatura_C': 'max','Vibracion_mm_s':'max','Horas_Operativas':'mean','Horas_Hasta_Revision':'mean'}).reset_index()
print(info_equipos_agg)

      ID_Equipo  Tipo_Equipo  Fabricante  Modelo  Potencia_kW  \
0             1            0           0       8        432.0   
1             1            0           0       8        432.0   
2             1            0           0       8        432.0   
3             1            0           0       8        432.0   
4             1            0           0       8        432.0   
...         ...          ...         ...     ...          ...   
4063        500            3           2       8        299.0   
4064        500            3           2       8        299.0   
4065        500            3           2       8        299.0   
4066        500            3           3       8        477.0   
4067        500            3           3       8        477.0   

      Horas_Recomendadas_Revision  Tipo_Mantenimiento  Quincena  \
0                           657.0                   0         1   
1                           657.0                   0         2   
2                 

Seleccionaremos las variables relevantes y si es necesario, generamos nuevas variables. Para el caso de Predección del fallo en 15 días con un 80% de fiabilidad

In [27]:
# Supongamos que tu DataFrame se llama df y tu variable objetivo es 'target'
correlation_matrix = info_equipos_agg.corr()
print(correlation_matrix['Tipo_Mantenimiento'].sort_values(ascending=False))


Tipo_Mantenimiento             1.000000
Horas_Operativas               0.725045
Temperatura_C                  0.087204
Vibracion_mm_s                 0.077410
Tipo_Equipo                    0.011587
ID_Equipo                      0.010826
Potencia_kW                    0.002130
Fabricante                    -0.008344
Modelo                        -0.008938
Horas_Recomendadas_Revision   -0.016464
Quincena                      -0.050462
Costo_Mantenimiento           -0.076438
Horas_Hasta_Revision          -0.715585
Name: Tipo_Mantenimiento, dtype: float64


In [28]:
# Supongamos que tu DataFrame se llama df y tu variable objetivo es 'target'
X = info_equipos_agg.drop('Tipo_Mantenimiento', axis=1)
y = info_equipos_agg['Tipo_Mantenimiento']

# Seleccionar las mejores características
selector = SelectKBest(score_func=f_classif, k=6)  # Cambia k por el número de características que deseas seleccionar
X_new = selector.fit_transform(X, y)

# Obtener los nombres de las mejores características
best_features = X.columns[selector.get_support()]
print(best_features.tolist())


['Quincena', 'Costo_Mantenimiento', 'Temperatura_C', 'Vibracion_mm_s', 'Horas_Operativas', 'Horas_Hasta_Revision']


In [35]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Supongamos que tu DataFrame original se llama info_equipos_agg
scaler = StandardScaler()
scaled_data = scaler.fit_transform(info_equipos_agg)

# Aplicar PCA para reducir dimensiones conservando el 95% de la varianza
pca = PCA(n_components=0.95)
pca_data = pca.fit_transform(scaled_data)

# Convertir los datos transformados a un DataFrame
pca_df = pd.DataFrame(pca_data, columns=[f'PC{i+1}' for i in range(pca_data.shape[1])])

# Mostrar la varianza explicada por cada componente principal
print("Varianza explicada por cada componente principal:")
print(pca.explained_variance_ratio_)

# Obtener las cargas de los componentes principales
loadings = pca.components_.T * np.sqrt(pca.explained_variance_)

# Crear un DataFrame con las cargas
loadings_df = pd.DataFrame(loadings, index=info_equipos_agg.columns, columns=[f'PC{i+1}' for i in range(pca_data.shape[1])])

# Mostrar las cargas de los componentes principales
print("Cargas de los componentes principales:")
print(loadings_df)

# Identificar las columnas más importantes para cada componente principal
important_features = loadings_df.abs().idxmax()
print("Columnas más importantes para cada componente principal:")
print(important_features)


Varianza explicada por cada componente principal:
[0.21908835 0.16688474 0.0867108  0.08536997 0.07790424 0.07581213
 0.07103963 0.06679579 0.05587366 0.04958134]
Cargas de los componentes principales:
                                  PC1       PC2       PC3       PC4       PC5  \
ID_Equipo                   -0.006649 -0.028229 -0.256811  0.137868  0.611429   
Tipo_Equipo                  0.004784  0.007219 -0.458784 -0.451748  0.228898   
Fabricante                   0.004951  0.021501  0.671219  0.052204 -0.063737   
Modelo                       0.002539 -0.010071 -0.094347  0.674916 -0.301182   
Potencia_kW                  0.019898  0.004555  0.512554 -0.538490  0.060453   
Horas_Recomendadas_Revision -0.050029 -0.121610  0.350881  0.362892  0.680285   
Tipo_Mantenimiento           0.708787  0.476427 -0.014095  0.027428  0.088300   
Quincena                    -0.298213  0.667689  0.022932  0.031447  0.065810   
Costo_Mantenimiento          0.448516 -0.608369  0.023699  0.016946 -

Probamos diferentes modelos, ajustamos hiperparametros y seguimos probando para evaluar si una máquina fallará en los próximos 15 días, con un 80% de fiabilidad

In [36]:
# Separar las características (X) y el objetivo (y)
X = info_equipos_agg[['Quincena', 'Costo_Mantenimiento', 'Temperatura_C', 'Vibracion_mm_s', 'Horas_Operativas', 'Horas_Hasta_Revision']]
y = info_equipos_agg['Tipo_Mantenimiento']

# Dividir los datos en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Escalar las características
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [37]:

# Crear y entrenar múltiples modelos
models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
    "Support Vector Machine": SVC(random_state=42),
    "Neural Network": MLPClassifier(random_state=42)
}

results = {}

for model_name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    results[model_name] = {
        "Accuracy": accuracy,
        "F1 Score": f1
    }

# Mostrar los resultados de cada modelo
for model_name in results:
    print(f"Resultados para {model_name}:")
    print(f"Accuracy: {results[model_name]['Accuracy'] * 100:.2f}%")
    print(f"Puntuación F1: {results[model_name]['F1 Score'] * 100:.2f}%")
    print("\n")


Resultados para Random Forest:
Accuracy: 97.79%
Puntuación F1: 98.10%


Resultados para Gradient Boosting:
Accuracy: 97.42%
Puntuación F1: 97.78%


Resultados para Support Vector Machine:
Accuracy: 94.96%
Puntuación F1: 95.77%


Resultados para Neural Network:
Accuracy: 97.30%
Puntuación F1: 97.69%




/anaconda/envs/azureml_py38/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [38]:
# Crear y entrenar múltiples modelos con ajuste de hiperparámetros
models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}

param_grids = {
    "Random Forest": {
        'n_estimators': [50, 100],
        'max_depth': [None, 10],
        'min_samples_split': [5]
    },
    "Gradient Boosting": {
        'n_estimators': [50],
        'learning_rate': [0.01],
        'max_depth': [2]
    }
}

results = {}

for model_name in models:
    model = models[model_name]
    param_grid = param_grids[model_name]
    
    grid_search = GridSearchCV(model, param_grid, cv=2)  # Reducir el número de divisiones a 2
    grid_search.fit(X_train_scaled, y_train)
    
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test_scaled)
    
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    results[model_name] = {
        "Accuracy": accuracy,
        "F1 Score": f1,
        "Best Params": grid_search.best_params_
    }

# Mostrar los resultados de cada modelo
for model_name in results:
    print(f"Resultados para {model_name}:")
    print(f"Accuracy: {results[model_name]['Accuracy'] * 100:.2f}%")
    print(f"Puntuación F1: {results[model_name]['F1 Score'] * 100:.2f}%")
    print(f"Mejores Parámetros: {results[model_name]['Best Params']}")
    print("\n")


Resultados para Random Forest:
Accuracy: 97.91%
Puntuación F1: 98.20%
Mejores Parámetros: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 100}


Resultados para Gradient Boosting:
Accuracy: 93.00%
Puntuación F1: 93.99%
Mejores Parámetros: {'learning_rate': 0.01, 'max_depth': 2, 'n_estimators': 50}




In [47]:
# Definir el modelo y los hiperparámetros a buscar
model = GradientBoostingClassifier(random_state=42)
param_grid = {
        'n_estimators': [18],
        'learning_rate': [0.01],
        'max_depth': [2]
    }
# Realizar la búsqueda en la cuadrícula con validación cruzada
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

# Obtener el mejor modelo de la búsqueda en la cuadrícula
best_model = grid_search.best_estimator_

# Hacer predicciones en el conjunto de prueba
y_pred = best_model.predict(X_test)

# Calcular la precisión y la puntuación F1
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Mejor Modelo: {best_model}")
print(f"Precisión: {accuracy}")
print(f"Puntuación F1: {f1}")

if accuracy >= 0.80 and f1 >= 0.75:
    print("El modelo cumple con los criterios de rendimiento requeridos.")
else:
    print("El modelo no cumple con los criterios de rendimiento requeridos.")


Mejor Modelo: GradientBoostingClassifier(learning_rate=0.01, max_depth=2, n_estimators=18,
                           random_state=42)
Precisión: 0.8636363636363636
Puntuación F1: 0.8923375363724539
El modelo cumple con los criterios de rendimiento requeridos.
